# **Thames Valley Crime Data Analysis**
## Statistical Testing - Summer vs Winter
---

### Objectives

Test whether the difference in crime between summer and winter (which we found out in `02_EDA.ipynb`) is **statistically significant**, or could be down to chance (Key Question 4).

### Planned Steps

1. Load and check the data
2. Probability of a crime in each season
3. Significance and choice of test
4. Normality check (Shapiro-Wilk)
5. t-test and conclusion

### Inputs

`../data/clean/crime_summary.csv`

### Key Questions explored in this notebook

4. Is there a seasonal pattern, and if so is the difference between summer and winter statistically significant?

### Hypothesis

**Practical hypothesis:** Crime in the Thames Valley is higher in summer than in winter. If this is true, police resources and awareness campaigns could be planned around the summer peak to mitigate against crime being committed.

**Statistical hypotheses:**

- **Null hypothesis (H0):** there is no difference in average monthly crimes between summer and winter.
- **Alternative hypothesis (H1):** average monthly crimes differ between summer and winter.

H1 is two-sided (summer and winter differ in either direction). If H0 is rejected, the averages from the EDA show which season is higher.

### Outputs

- t-test results and conclusion

---
### Setup

Import required libraries:

- `pandas` - to load and group the data.
- `pingouin` - used for the statistical tests.

**Note before running the notebook:**

- `pingouin` is installed from `requirements.txt` instead of in this notebook, so the right version is used. Remove the `#` from the notebook-only packages, then run `pip install -r requirements.txt`.
- Use `pingouin==0.5.4`. The newest version upgrades `pandas` and `numpy`, which breaks the other packages (see Bugs in README).

In [2]:
# Import required libraries
import pandas as pd
import pingouin as pg

print("Libraries imported successfully")

Libraries imported successfully


---
### Statistical concepts we will explore

In notebook `02_EDA.ipynb` (section 2) we looked at **mean, median, mode, standard deviation** and **variance**. In this notebook we will explore 3 more ideas:

- **Probability:** how likely something is to happen, from 0 (impossible) to 1 (certain).

- **Normal distribution:** a bell shape spread of data and t-testing, which assumes the data is normal.

- **Hypothesis testing:** checks if a difference is real or could be down to chance, measured using a p-value.

All the above let us describe data fairly and check the pattern is real, instead of making assumptions.

---
### 1. Load and check the data

We will load the data and add up the total number of crimes for each month, then split them into two groups **summer** and **winter**. The data contains 3 years of data, with each season being three months this will then give 9 months for each, so they can be compared fairly.

- **Summer** June, July and August
- **Winter** December, January and Ferbuary

We will do this becuase the t-test compares average monthly crimes in each season, so each month needs to be one value. When this is done we can then double-check the averages match `02_EDA.ipynb`.

In [5]:
# Load the data
summary = pd.read_csv("../data/clean/crime_summary.csv", parse_dates=["Month"])

# Total crimes for each month
season_monthly = summary.groupby(["Month", "Season"])["Count"].sum().reset_index()

# Season totals
summer_counts = season_monthly[season_monthly["Season"] == "Summer"]["Count"]
winter_counts = season_monthly[season_monthly["Season"] == "Winter"]["Count"]

print("Summer months:", len(summer_counts), "- average:", round(summer_counts.mean()))
print("Winter months:", len(winter_counts), "- average:", round(winter_counts.mean()))

Summer months: 9 - average: 17560
Winter months: 9 - average: 14458


This shows that the load and calculation worked and we now have 9 x Summer months averaging 17560 and 9 x
Winter months averaging 14458. This has been checked against the figures in `02_EDA.ipynb` and they match.

---
### 2. Probability of a crime in each season

Here we will look at if we picked a crime at random, what the probability is that it happened in each season?

Because each season covers 9 of the 36 months, if the season made no difference each would be **0.25** chance, so 1 in 4.

We will work this out by using the formula:

**P(A) = number of times A happens / number of outcomes**

In [7]:
# Probability of summer vs winter crimes
season_totals = summary.groupby("Season")["Count"].sum()
season_probability = (season_totals / season_totals.sum()).round(2)

season_probability

Season
Autumn    0.25
Spring    0.25
Summer    0.27
Winter    0.22
Name: Count, dtype: float64

From the result above, if we chose a random crime both autumn and spring the probability is 0.25 which is the equal share. However the probability of a crime in summer is 0.27, and 0.22 in winter - which supports the seasonal pattern.

**Note:** the numbers above equal 0.99, this will be due to the rounding of the numbers in the code cell above.

---
### 3. Significance and choice of test

There are lots of different types of statistical tests, the right one depends on the type of data and what we are looking to compare (see [Methodology Hub - Statistical Tests](https://methodologyhub.com/methods/statistics/statistical-tests)).

<small>Credit: Methodology Hub (2026). *Statistical Tests: Types, Uses, and How to Choose*. Retrieved 19 November 2026, from [https://methodologyhub.com/methods/statistics/statistical-tests/](https://methodologyhub.com/methods/statistics/statistical-tests/)</small>

<br>

Our hypotheses are at the top of this notebook. The test uses two probabilities:

- **Alpha:** 0.05 - We accept a 5% probability of finding a difference that doesn't exist.
- **P-Value:** The probability of seeing a difference this big if there were really no difference - if it is less than 0.05, we reject the null hypothesis.

The plan is to use a t-test which compares the average of two groups, though we need to first ensure that both groups have a **normal distribution** (bell shaped). If they are not normal, we will use the **Mann-Whitney** test instead.